# FPL Hidden Gem Finder — EDA & Feature Engineering

**Phases:**
1. Load & inspect raw data
2. Clean data (`src/data_cleaning.py`)
3. Feature engineering (`src/feature_engineering.py`)
4. Modeling (`src/model.py`)
5. Save processed data for the Streamlit app

## 1. Load & inspect raw data

Check the actual columns before assuming anything — verify whether this is a season-aggregated file or gameweek-by-gameweek.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd

df = pd.read_csv('../data/raw/YOUR_FILE.csv')
print(df.columns.tolist())
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe()

## 2. Clean data

Implement `load_and_clean_data` in `src/data_cleaning.py`, then call it here.

In [ ]:
from src.data_cleaning import load_and_clean_data

df_clean = load_and_clean_data('../data/raw/YOUR_FILE.csv')
df_clean.head()

## 3. Feature engineering

Implement each function in `src/feature_engineering.py`, then apply here. Remember the leakage warning on rolling features (shift before rolling).

In [ ]:
from src.feature_engineering import (
    add_points_per_million,
    add_rolling_form,
    add_custom_fixture_difficulty,
    add_minutes_reliability,
)

df_feat = add_points_per_million(df_clean)
df_feat = add_rolling_form(df_feat, window=3)
df_feat = add_custom_fixture_difficulty(df_feat)
df_feat = add_minutes_reliability(df_feat, window=5)
df_feat.head()

## 4. EDA — sanity checks & visuals

Before modeling: plot price vs points, check feature distributions, sanity-check the fixture difficulty metric against a few known easy/hard fixtures.

In [ ]:
import plotly.express as px

# TODO: scatter plot price vs total points, colored by position

## 5. Modeling

Implement `src/model.py`, then train/evaluate here. Use a **time-based split**, never random shuffle.

In [ ]:
from src.model import time_based_split, train_baseline_model, train_linear_regression, evaluate_model

train_df, test_df = time_based_split(df_feat, train_gw_end=25, test_gw_start=26)

features = ['rolling_form_3gw', 'minutes_reliability', 'custom_fixture_difficulty', 'was_home']
target = 'total_points'

model = train_linear_regression(train_df, features, target)
results = evaluate_model(model, test_df, features, target)
print(results)

**Interpretation:** _(write 2-3 sentences here on why R²/MAE are what they are — football randomness, injuries not in the data, etc. This matters more than the score itself.)_

## 6. Save processed data for the Streamlit app

In [ ]:
df_feat.to_parquet('../data/processed/fpl_processed.parquet', index=False)